# Lecture 1 — Introduction and Linear Classification

This notebook turns Lecture 1 into an executable learning artifact. We move from the machine-learning problem to a linear hypothesis class, visualize its geometry, measure training error, and implement the perceptron update from scratch.

**Learning order:** mathematics → calculation → visualization → implementation → connection to Lecture 2.

## 1. Binary classification

We are given labeled examples $(x_i,y_i)$ with $y_i\in\{-1,+1\}$. The goal is to learn a classifier that works on unseen examples, not merely memorize the training set.

A linear classifier uses the score

$$s(x)=\theta^Tx$$

and predicts

$$f(x;\theta)=\operatorname{sign}(\theta^Tx).$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[4,4],[5,3],[3,5],[1,1],[2,1],[1,2]], dtype=float)
y = np.array([1,1,1,-1,-1,-1])
theta = np.array([1.0, 1.0])

scores = X @ theta
predictions = np.where(scores >= 0, 1, -1)

print('scores:     ', scores)
print('predictions:', predictions)
print('labels:     ', y)

## 2. Decision boundary and geometry

The prediction changes when the score crosses zero, so the decision boundary is

$$\theta^Tx=0.$$

For two features this is $\theta_1x_1+\theta_2x_2=0$, a line through the origin. The vector $\theta$ is perpendicular to the boundary. Positive scores lie on one side and negative scores on the other.

In [ ]:
positive = y == 1
negative = y == -1
x1 = np.linspace(0, 6, 200)
x2 = -(theta[0] / theta[1]) * x1

plt.figure(figsize=(7,6))
plt.scatter(X[positive,0], X[positive,1], marker='+', s=150, label='label +1')
plt.scatter(X[negative,0], X[negative,1], marker='_', s=150, label='label -1')
plt.plot(x1, x2, label=r'$\theta^T x=0$')
plt.arrow(0, 0, theta[0], theta[1], head_width=0.12, length_includes_head=True)
plt.xlim(0,6); plt.ylim(0,6)
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.title('Linear classifier through the origin')
plt.grid(True, alpha=0.25); plt.legend(); plt.show()

## 3. Training error and agreement

The zero-one training error is

$$\widehat E(\theta)=\frac{1}{n}\sum_{i=1}^n\mathbf{1}[f(x_i;\theta)\ne y_i].$$

For an individual example, the agreement is $y_i\theta^Tx_i$. Positive agreement means correct classification; negative agreement means a mistake.

In [ ]:
def predict(X, theta):
    return np.where(X @ theta >= 0, 1, -1)

def training_error(X, y, theta):
    return np.mean(predict(X, theta) != y)

print('training error:', training_error(X, y, theta))
print('agreements:    ', y * (X @ theta))

## 4. Perceptron update from scratch

When $(x_i,y_i)$ is misclassified, the perceptron updates

$$\theta\leftarrow\theta+y_ix_i.$$

If the example is wrong, then $y_i\theta^Tx_i<0$. After the update,

$$y_i(\theta+y_ix_i)^Tx_i=y_i\theta^Tx_i+\lVert x_i\rVert^2.$$

Thus the agreement for the current example increases. Other examples can change in either direction.

In [ ]:
theta_before = np.array([-1.0, 1.0])
x_i = np.array([4.0, 4.0])
y_i = 1

before = y_i * (theta_before @ x_i)
theta_after = theta_before + y_i * x_i
after = y_i * (theta_after @ x_i)

print('theta before:', theta_before)
print('agreement before:', before)
print('theta after: ', theta_after)
print('agreement after: ', after)
print('increase:', after - before)
print('||x_i||^2:', x_i @ x_i)

## 5. Full perceptron implementation

The implementation below uses only NumPy. A correct example leaves the parameters unchanged; a mistake triggers the update.

In [ ]:
def perceptron(X, y, epochs=20):
    theta = np.zeros(X.shape[1], dtype=float)
    errors = []
    updates = 0

    for epoch in range(epochs):
        for x_i, y_i in zip(X, y):
            prediction = 1 if theta @ x_i >= 0 else -1
            if prediction != y_i:
                theta += y_i * x_i
                updates += 1
        errors.append(training_error(X, y, theta))
        if errors[-1] == 0:
            break
    return theta, errors, updates

theta_learned, errors, updates = perceptron(X, y)
print('learned theta:', theta_learned)
print('updates:', updates)
print('training errors:', errors)

In [ ]:
positive = y == 1
negative = y == -1
x1 = np.linspace(0, 6, 200)
x2 = -(theta_learned[0] / theta_learned[1]) * x1

plt.figure(figsize=(7,6))
plt.scatter(X[positive,0], X[positive,1], marker='+', s=150, label='label +1')
plt.scatter(X[negative,0], X[negative,1], marker='_', s=150, label='label -1')
plt.plot(x1, x2, label=r'learned $\theta^T x=0$')
plt.xlim(0,6); plt.ylim(0,6)
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.title('Decision boundary after perceptron training')
plt.grid(True, alpha=0.25); plt.legend(); plt.show()

plt.figure(figsize=(7,4))
plt.plot(range(1, len(errors)+1), errors, marker='o')
plt.xlabel('Epoch'); plt.ylabel('Training error')
plt.title('Perceptron training error')
plt.grid(True, alpha=0.25); plt.show()

## 6. A limitation of the hypothesis class: XOR

XOR is not linearly separable. A single straight decision boundary cannot classify all four points correctly. This is a limitation of the model family, not merely a bad parameter choice.

In [ ]:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([-1,1,1,-1])
theta_xor, xor_errors, xor_updates = perceptron(X_xor, y_xor, epochs=20)
print('theta after 20 epochs:', theta_xor)
print('training errors:', xor_errors)

## 7. Bias and augmented vectors

The through-origin classifier can be generalized with a bias:

$$f(x)=\operatorname{sign}(\theta^Tx+\theta_0).$$

The boundary is $\theta^Tx+\theta_0=0$. With

$$\widetilde x=\begin{bmatrix}x\\1\end{bmatrix},\qquad\widetilde\theta=\begin{bmatrix}\theta\\\theta_0\end{bmatrix},$$

we have $\widetilde\theta^T\widetilde x=\theta^Tx+\theta_0$. Lecture 2 develops this formulation more fully.

In [ ]:
x = np.array([4.0, 4.0])
theta = np.array([1.0, 2.0])
theta_0 = -5.0

x_aug = np.append(x, 1.0)
theta_aug = np.append(theta, theta_0)

print('original score:  ', theta @ x + theta_0)
print('augmented score:', theta_aug @ x_aug)

## 8. Takeaways

You should be able to explain why generalization matters, what a hypothesis class is, how $\theta^Tx$ determines a prediction, why $\theta^Tx=0$ is the decision boundary, why $\theta$ is normal to that boundary, how training error is measured, and why the perceptron update is $\theta\leftarrow\theta+y_ix_i$.

The course progression is:

```text
Lecture 1 → linear classification + perceptron update
Lecture 2 → perceptron convergence + fuller implementation
Lecture 3 → loss + regularization + optimization
Lecture 4 → validation + cross-validation + hyperparameter selection
```